In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utils import bootstrap_project_paths

project_root = bootstrap_project_paths()

from src.data import (
    load_sales, load_web_traffic, 
    load_promotions, load_order_items, 
    load_orders
)
from src.features import add_time_features, add_lag_features, list_feature_columns, one_hot_encode
from src.pipelines import fit_and_predict, train_validate_models
from src.models import SklearnRegressorConfig, SklearnRegressorWrapper
import pandas as pd


DATA_ROOT = project_root / "data/datathon-2026-round-1"

In [ ]:
TRAIN_RANGE = ("2013-01-01", "2022-12-31")
PREDICT_RANGE = ("2023-01-01", "2024-07-01")
df_sales = load_sales(data_root=DATA_ROOT)

In [ ]:
df = pd.DataFrame(
   {"date": pd.date_range(start=TRAIN_RANGE[0], end=PREDICT_RANGE[1], freq="D")}
)
df = df.merge(df_sales, on="date", how="left")
df = add_time_features(df, date_col="date")
# add feature year_after_shock = max(0, year - 2019)
# add feature shock_period = 1 if date is between 2019-03-01 and 2022-12-31, else 0

# không thể dùng year after shock do model chưa gặp data nào khi hồi phục
# df["year_after_shock"] = (df["date"].dt.year - 2019).clip(lower=0)
df["shock_period"] = ((df["date"] >= "2019-03-01") & (df["date"] <= "2022-12-31")).astype(int)
# df["month_after_shock"] = ((df["date"].dt.year - 2019) * 12 + df["date"].dt.month - 3).clip(lower=0)
# df["year_distance_from_2019"] = (df["date"].dt.year - 2019).abs()
# year dist from shock giúp validation tốt hơn nhưng test lại tệ

# List all feature columns
feature_columns = list_feature_columns(df)
print(feature_columns["all_features"])
df.tail()

# Cần giải thích tại sao lag 30 cho revenue lại tệ
# 11 mạnh vượt trội, 12, 13, 14 cải thiện nhưng không lấy 
# 21, 28, 30, 10 làm tệ đi
# COGS giải thích tốt hơn revenue trên tập test, vượt trội
df = add_lag_features(df, lags=[1,7], target_col="COGS", date_col="date")
# df = add_lag_features(df, lags=[1,7,11], target_col="Revenue", date_col="date")
# Cái feature này thần kỳ vcl
df["is_month8_odd"] = ((df["month"] == 8) & (df["year"] % 2 == 1))

df["is_peak_period"] = ((df["day"] <= 5) | (df["day"] >= 25)).astype(int)
df["is_peak_period_lag7"] = df["is_peak_period"].shift(7)
df["is_peak_period_lag1"] = df["is_peak_period"].shift(1)

In [ ]:
# Holiday features
fixed_holidays = [
    {"day": 1, "month": 1, "name": "New Year's Day"},
    {"day": 30, "month": 4, "name": "Victory Day"},
    {"day": 1, "month": 5, "name": "International Labor Day"},
    {"day": 2, "month": 9, "name": "National Day"},
]

years = df["date"].dt.year.dropna().unique().tolist()
fixed_dates = []
for y in years:
    for h in fixed_holidays:
        fixed_dates.append(pd.Timestamp(year=int(y), month=h["month"], day=h["day"]))
fixed_dates = pd.to_datetime(fixed_dates)
holiday_dates = pd.to_datetime(pd.Index(fixed_dates))

df["is_holiday"] = df["date"].isin(holiday_dates).astype(int)

model_config = SklearnRegressorConfig(
    model_type="lightgbm"
)

In [ ]:
model_config = SklearnRegressorConfig(
    model_type="lightgbm",
)

result = train_validate_models(
    df=df,
    model_config=model_config,
    train_range=("2013-01-01", "2021-12-31"),
    predict_range=("2022-01-01", "2022-12-31"),
    importance_plot=True,
)

result["metrics"]



In [ ]:
model_config = SklearnRegressorConfig(
    model_type="lightgbm",
)

submission = fit_and_predict(
    df=df,
    model_config=model_config,
    train_range=TRAIN_RANGE,
    predict_range=PREDICT_RANGE,
)
submission.head()
submission_name = "miracle13"
submission.to_csv(project_root / f"submissions/{submission_name}.csv", index=False)